# Personal Information
Name: **Jia Long Bao**

StudentID: **12593400**

Email: [**jialong.bao@student.uva.nl**](jialong.bao@student.uva.nl)

Submitted on: **XX.10.2025**

# Data Context
The data used in this research originates from Rijden De Treinen, capturing spatial, temporal, and reported causes that influence train delays in the Netherlands. 

The spatial data, which includes (station locations and) railway connections, is extracted from (the NS API and) Rijden de Treinen, a platform that processes real-time open data from NDOV (Nationale Data Openbaar Vervoer). This ensures an accurate representation of the railway network, which forms the foundation of modeling the graph structure. Additionally, annual passenger volume per station, retrieved from the NS dashboard, provides insights into station-level demand.

The temporal data consists of train service schedules, recorded delays and disruptions all sourced from Rijden de Treinen.

Github: https://github.com/jbaonl/MSc-Thesis

In [ ]:
import pandas as pd
from pathlib import Path

# Base directory for all data
base_dir = Path("downloads/data")

# Define subfolder names
folders = {
    "disruptions": base_dir / "NS-disruptions",
    "services": base_dir / "NS-services",
    "stations": base_dir / "NS-stations",
    "tariff": base_dir / "NS-tariff-distances"
}

# Function to safely read CSVs (handles gzipped and standard files)
def read_all_csvs_from_folder(folder_path):
    csv_files = sorted(folder_path.glob("*.csv*"))
    dfs = []
    for f in csv_files:
        try:
            if f.suffix == ".gz":
                df = pd.read_csv(f, compression="gzip", low_memory=False)
            else:
                df = pd.read_csv(f, low_memory=False)
            df["source_file"] = f.name  # tag origin
            dfs.append(df)
        except Exception as e:
            print(f"⚠️ Skipped {f.name}: {e}")
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        return pd.DataFrame()

# Read all groups
data = {}
for key, folder in folders.items():
    print(f"📂 Reading {key} data from {folder} ...")
    data[key] = read_all_csvs_from_folder(folder)
    print(f"   → {len(data[key])} rows, {len(data[key].columns)} columns")

# Optional: quick check of available datasets
for key, df in data.items():
    print(f"\n✅ {key.upper()} sample:")
    print(df.head(2))


📂 Reading disruptions data from downloads\data\NS-disruptions ...
   → 31895 rows, 15 columns
📂 Reading services data from downloads\data\NS-services ...
⚠️ Skipped services-2019.csv: Error tokenizing data. C error: out of memory
⚠️ Skipped services-2019.csv.gz: Error tokenizing data. C error: out of memory
⚠️ Skipped services-2020.csv: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.


In [3]:
pd.read_csv("downloads/data/NS-services/services-2019.csv.gz", compression="gzip", low_memory=False).head(2)

ParserError: Error tokenizing data. C error: out of memory

In [1]:
import pandas as pd
from pathlib import Path

# Define folders
services_dir = Path("downloads/data/NS-services")
parquet_dir = services_dir / "parquet"
parquet_dir.mkdir(parents=True, exist_ok=True)

# 🔍 Find ONLY .csv.gz files
gz_files = sorted(services_dir.glob("*.csv.gz"))
print(f"Found {len(gz_files)} compressed service files to convert.\n")

for f in gz_files:
    try:
        # Read compressed CSV
        df = pd.read_csv(f, compression="gzip", low_memory=False)

        # Convert name -> .parquet (remove .gz)
        parquet_file = parquet_dir / f.with_suffix("").with_suffix(".parquet").name

        # Write to Parquet
        df.to_parquet(parquet_file, index=False)
        print(f"✅ Converted {f.name} → {parquet_file.name}")
    except Exception as e:
        print(f"⚠️ Skipped {f.name}: {e}")


Found 6 compressed service files to convert.

⚠️ Skipped services-2019.csv.gz: Error tokenizing data. C error: out of memory


KeyboardInterrupt: 